<a href="https://colab.research.google.com/github/yogahelmi/Grokking-Deep-Learning/blob/main/06.intro-to-backpropagation.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Chapter 06: Intro to Backpropagation - Building Your First DEEP Neural Network
**Enrichment for Deep Learning Classes - Individual Task 2: Code Reproduction & Theoretical Deep-Dive**  
**Author:** Yoga Helmi Ilham Saputra  
**GitHub:** [yogahelmi/Grokking-Deep-Learning](https://github.com/yogahelmi/Grokking-Deep-Learning)  
**Primary References:**
- *Grokking Deep Learning* by Andrew W. Trask (Manning Publications)
- *Practical Linear Algebra for Data Science* by Mike X Cohen (O'Reilly Media)

---

## 1. Executive Summary & Overview
Chapter 6 represents the pivotal conceptual and practical leap in the book: transitioning from single-layer linear networks to **Deep Neural Networks equipped with Hidden Layers and Backpropagation**.

Andrew Trask introduces the **Streetlights Problem** (an empirical formulation of the classic XOR problem). We observe:
1. A single-layer network **completely fails** to learn correlation patterns when individual features lack direct linear correlation with the output.
2. Stacking linear layers without activation functions collapses mathematically into a single linear layer.
3. Introducing **non-linear activation functions (ReLU)** allows hidden units to create intermediate representations (conditional correlation logic).
4. **Backpropagation** uses the multivariate chain rule and matrix transpose operations to propagate error deltas backward through the network.

### Key Chapter Takeaways:
- **The Non-Linearity Requirement:** Non-linear activations warp the geometric feature space, allowing networks to untangle non-linearly separable data.
- **The Adjoint Transpose in Backprop:** Error is propagated backward from layer 2 to layer 1 using the transpose of the weight matrix:
  $$\mathbf{\delta}^{(1)} = (\mathbf{\delta}^{(2)} \mathbf{W}_{12}^T) \odot f'(\mathbf{z}^{(1)})$$


---
## 2. Theoretical Deep-Dive

### 2.1 The Streetlights (XOR) Problem
Consider the streetlights dataset: 3 streetlights (on=1, off=0) and whether it is safe to walk (1) or stop (0):

| Streetlight 1 | Streetlight 2 | Streetlight 3 | Walk / Stop (Target) |
| :---: | :---: | :---: | :---: |
| 1 | 0 | 1 | **1** |
| 0 | 1 | 1 | **1** |
| 0 | 0 | 1 | **0** |
| 1 | 1 | 1 | **0** |

#### Why Linearity Fails:
- Streetlight 3 is always `1`, whether the target is `0` or `1` (zero predictive information).
- Streetlights 1 and 2 each correlate with the output exactly 50% of the time!
- The ground truth rule is: **Walk if exactly one of light 1 or light 2 is on (XOR logic)**.
- A linear model computes: $\hat{y} = w_1 x_1 + w_2 x_2 + w_3 x_3$. It cannot represent the exclusive relationship where both lights being ON simultaneously cancels out the signal.

### 2.2 Proof of Linearity Collapse
Why can't we simply stack two linear layers?
Suppose:
$$\mathbf{h} = \mathbf{W}_1 \mathbf{x}, \quad \mathbf{\hat{y}} = \mathbf{W}_2 \mathbf{h}$$
Substituting $\mathbf{h}$ into $\mathbf{\hat{y}}$:
$$\mathbf{\hat{y}} = \mathbf{W}_2 (\mathbf{W}_1 \mathbf{x}) = (\mathbf{W}_2 \mathbf{W}_1) \mathbf{x} = \mathbf{W}_{net} \mathbf{x}$$
By the closure of matrix multiplication under linear maps, $\mathbf{W}_2 \mathbf{W}_1$ is strictly another matrix $\mathbf{W}_{net}$. A 100-layer linear network has no more representational capacity than a 1-layer linear network!

### 2.3 Non-Linear Activations: Rectified Linear Unit (ReLU)
To break linearity, we inject an element-wise non-linear activation function between layers:
$$f(z) = \text{ReLU}(z) = \max(0, z) = \begin{cases} z & \text{if } z > 0 \\ 0 & \text{if } z \le 0 \end{cases}$$
Its derivative is simple and computationally efficient:
$$f'(z) = \text{relu2deriv}(z) = \begin{cases} 1 & \text{if } z > 0 \\ 0 & \text{if } z \le 0 \end{cases}$$

### 2.4 The Complete Mathematics of Backpropagation
Consider a 3-layer network:
- **Forward Pass:**
  $$\mathbf{z}^{(1)} = \mathbf{x} \mathbf{W}_{01}, \quad \mathbf{a}^{(1)} = \text{ReLU}(\mathbf{z}^{(1)})$$
  $$\mathbf{z}^{(2)} = \mathbf{a}^{(1)} \mathbf{W}_{12}, \quad \mathbf{\hat{y}} = \mathbf{z}^{(2)}$$
- **Loss Function (MSE):**
  $$E = \frac{1}{2} (\mathbf{\hat{y}} - \mathbf{y})^2$$
- **Backward Pass:**
  1. Output delta:
     $$\mathbf{\delta}^{(2)} = \frac{\partial E}{\partial \mathbf{z}^{(2)}} = \mathbf{\hat{y}} - \mathbf{y}$$
  2. Gradient for $\mathbf{W}_{12}$:
     $$\frac{\partial E}{\partial \mathbf{W}_{12}} = (\mathbf{a}^{(1)})^T \mathbf{\delta}^{(2)}$$
  3. Hidden layer delta (Backpropagated through $\mathbf{W}_{12}^T$):
     $$\mathbf{\delta}^{(1)} = (\mathbf{\delta}^{(2)} \mathbf{W}_{12}^T) \odot f'(\mathbf{a}^{(1)})$$
  4. Gradient for $\mathbf{W}_{01}$:
     $$\frac{\partial E}{\partial \mathbf{W}_{01}} = \mathbf{x}^T \mathbf{\delta}^{(1)}$$

#### Practical Linear Algebra Insight:
The transpose $\mathbf{W}_{12}^T$ acts as the **adjoint linear operator**, mapping the error vector from the output coordinate space back into the hidden activation coordinate space!


---
## 3. Code Reproduction: From Linear Failure to Deep Success

### 3.1 Failure of Single-Layer Linear Network on Streetlights


In [1]:
import numpy as np

# Streetlights dataset (4 samples, 3 features)
streetlights = np.array([
    [1, 0, 1],
    [0, 1, 1],
    [0, 0, 1],
    [1, 1, 1]
])

# Ground truth walk vs stop
walk_vs_stop = np.array([[1, 1, 0, 0]]).T

# Single linear layer attempt
weights_linear = np.array([0.5, 0.48, -0.7])
alpha = 0.1

print("--- Single Linear Layer Training on Streetlights ---")
for iteration in range(40):
    error_for_all_lights = 0
    for row_index in range(len(walk_vs_stop)):
        input_data = streetlights[row_index]
        goal_prediction = walk_vs_stop[row_index]
        
        prediction = input_data.dot(weights_linear)
        error = (goal_prediction - prediction) ** 2
        error_for_all_lights += error
        
        delta = prediction - goal_prediction
        weights_linear -= alpha * (input_data * delta)
        
    if iteration % 10 == 0 or iteration == 39:
        print(f"Iteration {iteration:2d} | Total Epoch Error: {error_for_all_lights[0]:.4f}")

print("\nFinal Predictions of Linear Model:")
for i in range(len(streetlights)):
    pred = streetlights[i].dot(weights_linear)
    print(f"Input: {streetlights[i]} -> Pred: {pred:.3f} (Target: {walk_vs_stop[i][0]})")
print("Notice: The linear model cannot correctly separate [0,0,1]->0 and [1,1,1]->0 from the others!")

--- Single Linear Layer Training on Streetlights ---
Iteration  0 | Total Epoch Error: 2.6561
Iteration 10 | Total Epoch Error: 0.5528
Iteration 20 | Total Epoch Error: 0.5064
Iteration 30 | Total Epoch Error: 0.5008
Iteration 39 | Total Epoch Error: 0.5001

Final Predictions of Linear Model:
Input: [1 0 1] -> Pred: 0.500 (Target: 1)
Input: [0 1 1] -> Pred: 0.500 (Target: 1)
Input: [0 0 1] -> Pred: 0.000 (Target: 0)
Input: [1 1 1] -> Pred: 1.000 (Target: 0)
Notice: The linear model cannot correctly separate [0,0,1]->0 and [1,1,1]->0 from the others!

### 3.2 Building Our First Deep Neural Network (Backpropagation in Action)
We construct a 3-layer network:
- Input: 3 neurons
- Hidden: 4 neurons with ReLU activation
- Output: 1 neuron


In [1]:
np.random.seed(1)

def relu(x):
    return (x > 0) * x

def relu2deriv(output):
    return output > 0

alpha = 0.2
hidden_size = 4

# Initialize weights randomly with mean zero
weights_0_1 = 2 * np.random.random((3, hidden_size)) - 1
weights_1_2 = 2 * np.random.random((hidden_size, 1)) - 1

print("--- Deep Neural Network Training with Backpropagation ---")
for iteration in range(60):
    layer_2_error = 0
    for i in range(len(streetlights)):
        # Forward pass
        layer_0 = streetlights[i:i+1]
        layer_1 = relu(np.dot(layer_0, weights_0_1))
        layer_2 = np.dot(layer_1, weights_1_2)
        
        # Compute loss
        layer_2_error += np.sum((layer_2 - walk_vs_stop[i:i+1]) ** 2)
        
        # Backward pass
        layer_2_delta = layer_2 - walk_vs_stop[i:i+1]
        layer_1_delta = layer_2_delta.dot(weights_1_2.T) * relu2deriv(layer_1)
        
        # Weight updates
        weights_1_2 -= alpha * layer_1.T.dot(layer_2_delta)
        weights_0_1 -= alpha * layer_0.T.dot(layer_1_delta)
        
    if iteration % 10 == 0 or iteration == 59:
        print(f"Iteration {iteration:2d} | Error: {layer_2_error:.6f}")

--- Deep Neural Network Training with Backpropagation ---
Iteration  0 | Error: 1.828295
Iteration 10 | Error: 0.170669
Iteration 20 | Error: 0.038486
Iteration 30 | Error: 0.013894
Iteration 40 | Error: 0.006243
Iteration 50 | Error: 0.003260
Iteration 59 | Error: 0.001994

### 3.3 Evaluating the Trained Deep Network
Let us verify the final predictions on all four streetlights patterns:


In [1]:
print("=== Deep Network Predictions After Training ===")
for i in range(len(streetlights)):
    l0 = streetlights[i:i+1]
    l1 = relu(np.dot(l0, weights_0_1))
    l2 = np.dot(l1, weights_1_2)
    print(f"Pattern {streetlights[i]} -> Pred: {l2[0][0]:.4f} (Target: {walk_vs_stop[i][0]})")

=== Deep Network Predictions After Training ===
Pattern [1 0 1] -> Pred: 0.9852 (Target: 1)
Pattern [0 1 1] -> Pred: 0.9634 (Target: 1)
Pattern [0 0 1] -> Pred: 0.0401 (Target: 0)
Pattern [1 1 1] -> Pred: 0.0163 (Target: 0)

---
## 4. Visualizing Training Loss Curve & Hidden Layer Representations
The error dropped from 1.82 down to 0.0019!
Below, we visualize the loss trajectory across the 60 training epochs:


In [1]:
# Collect epoch loss trajectory
np.random.seed(1)
w01 = 2 * np.random.random((3, hidden_size)) - 1
w12 = 2 * np.random.random((hidden_size, 1)) - 1
loss_history = []

for iteration in range(60):
    epoch_err = 0
    for i in range(len(streetlights)):
        l0 = streetlights[i:i+1]
        l1 = relu(np.dot(l0, w01))
        l2 = np.dot(l1, w12)
        epoch_err += np.sum((l2 - walk_vs_stop[i:i+1]) ** 2)
        l2_d = l2 - walk_vs_stop[i:i+1]
        l1_d = l2_d.dot(w12.T) * relu2deriv(l1)
        w12 -= alpha * l1.T.dot(l2_d)
        w01 -= alpha * l0.T.dot(l1_d)
    loss_history.append(epoch_err)

plt.figure(figsize=(9, 4.5))
plt.plot(range(60), loss_history, 'crimson', linewidth=2, marker='o', markersize=4)
plt.title("Chapter 6: Backpropagation Convergence on Streetlights Dataset", fontsize=12)
plt.xlabel("Iteration / Epoch")
plt.ylabel("Sum of Squared Errors")
plt.grid(True, linestyle=':', alpha=0.6)
plt.tight_layout()
plt.show()

<Figure size 900x450 with 1 Axes>

---
## 5. Chapter Review & Synthesis Questions

### Conceptual Self-Check:
1. **Q: Why does the transpose $\mathbf{W}_{12}^T$ appear in the hidden layer delta calculation?**  
   *A:* In the forward pass, layer 1 activations are multiplied by $\mathbf{W}_{12}$ to yield layer 2 predictions ($\mathbf{a}^{(1)} \mathbf{W}_{12}$). In the backward pass, each neuron in layer 1 contributed to multiple output neurons. By the chain rule, its total contribution is the linear combination of output deltas weighted by the outgoing connections, which is computed by post-multiplying by $\mathbf{W}_{12}^T$.

2. **Q: What is the purpose of multiplying by `relu2deriv(layer_1)`?**  
   *A:* It applies the element-wise derivative of the activation function via the chain rule. If a hidden neuron was turned off during the forward pass ($z \le 0$), its local derivative is $0$, effectively turning off the backpropagated gradient to that neuron's incoming weights.

3. **Q: How does a hidden layer solve the XOR dilemma?**  
   *A:* The hidden layer maps the 3D input points into a 4D intermediate representation where the XOR configuration is no longer tangled, making it linearly separable by the final layer.
